# 🛡️ VoiceShield: Automated 2,000-Sample Mixed Bilingual Benchmark Pipeline
### Rigorous Out-of-Sample Performance Evaluation on NVIDIA T4 GPU (Google Colab)

This notebook automates testing on **2,000 brand-new, unseen audio samples** across all eligible categories:
- 🎙️ **1,000 Real Human Voices** (500 English + 500 Hindi) streamed directly from the official **test splits of Google FLEURS** (strict zero training leakage).
- 🤖 **1,000 AI Synthetic Voices** (500 English + 500 Hindi) synthesized via neural vocoders (`Guy`, `Jenny`, `Aria`, `Ryan`, `Neerja`, `Madhur`, `Swara`).
- 📡 **Realistic Acoustic Perturbations**: Simulates real-world phone line bandpass (300Hz-3400Hz), room reverberation / replay echo, and ambient background noise.
- ⏱️ **Strict Duration Constraint**: All audio files strictly bounded between **3.0s and 6.0s**.
- 📦 **Automated Golden Vault Ingestion**: If the model passes the quality threshold (Accuracy $\ge$ 88%, Precision $\ge$ 85%), it automatically archives the benchmark into `test-Bi-new-2k-01.zip` for one-click download.

In [ ]:
# =====================================================
# Step 0: Hardware Verification (NVIDIA T4 GPU Check)
# =====================================================
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# =====================================================
# Step 1: Install Required Dependencies
# =====================================================
!pip install -q edge-tts datasets soundfile librosa transformers evaluate accelerate scipy tqdm pandas
print("✅ All dependencies installed successfully!")


In [ ]:
# =====================================================
# Step 2: Acoustic Augmentations & Duration Enforcement
# =====================================================
import numpy as np
import soundfile as sf
import librosa
from scipy import signal

def apply_phone_bandpass(audio, sr=16000, lowcut=300.0, highcut=3400.0):
    """Simulates 300Hz-3400Hz narrowband telephonic line with dynamic range compression."""
    nyq = 0.5 * sr
    low = max(lowcut / nyq, 0.01)
    high = min(highcut / nyq, 0.99)
    b, a = signal.butter(4, [low, high], btype='band')
    filtered = signal.lfilter(b, a, audio)
    compressed = np.tanh(filtered * 1.4)
    max_val = np.max(np.abs(compressed))
    return (compressed / max_val * 0.9).astype(np.float32) if max_val > 0 else compressed.astype(np.float32)

def apply_room_reverb(audio, sr=16000, rt60=0.35):
    """Convolves audio with synthetic Room Impulse Response (RIR) simulating room echo/replay."""
    length = int(sr * rt60)
    decay = np.exp(-3.0 * np.linspace(0, 1, length))
    noise = np.random.normal(0, 1, length)
    rir = decay * noise
    rir[0] = 1.0
    rir = rir / np.sqrt(np.sum(rir ** 2))
    reverbed = signal.convolve(audio, rir, mode='same')
    max_val = np.max(np.abs(reverbed))
    return (reverbed / max_val * 0.9).astype(np.float32) if max_val > 0 else reverbed.astype(np.float32)

def apply_ambient_noise(audio, snr_db=22.0):
    """Injects calibrated ambient room air noise."""
    power = np.mean(audio ** 2)
    if power <= 0: return audio
    noise_power = power / (10 ** (snr_db / 10.0))
    noise = np.random.normal(0, np.sqrt(noise_power), len(audio))
    noisy = audio + noise
    max_val = np.max(np.abs(noisy))
    return (noisy / max_val * 0.9).astype(np.float32) if max_val > 0 else noisy.astype(np.float32)

def bound_duration(audio, sr=16000, min_sec=3.0, max_sec=5.5):
    """Ensures audio duration is strictly within [min_sec, max_sec]."""
    dur = len(audio) / sr
    if dur > max_sec:
        audio = audio[:int(sr * max_sec)]
    elif dur < min_sec:
        pad_len = int(sr * (min_sec + 0.2 - dur))
        audio = np.pad(audio, (0, pad_len), mode='constant')
    return audio.astype(np.float32)

print("✅ Acoustic signal processing operators ready.")


In [ ]:
# =====================================================
# Step 3: Automated Ingestion of 1,000 Real Human Audio Clips
# (500 English + 500 Hindi from FLEURS test split)
# =====================================================
import os
import glob
from datasets import load_dataset

os.makedirs("eval_2k_dataset/real", exist_ok=True)
os.makedirs("eval_2k_dataset/fake", exist_ok=True)

print("📥 1. Streaming 500 Real English clips from Google FLEURS test split...")
count_en = len(glob.glob("eval_2k_dataset/real/en_real_*.wav"))
if count_en < 500:
    ds_en = load_dataset("google/fleurs", "en_us", split="test", streaming=True)
    skipped = 0
    for item in ds_en:
        if skipped < 100:  # Skip first 100 to avoid any overlap with previous benchmarks
            skipped += 1
            continue
        if count_en >= 500: break
        arr = item["audio"]["array"]
        sr = item["audio"]["sampling_rate"]
        if sr != 16000: arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
        arr = bound_duration(arr, sr=16000, min_sec=3.0, max_sec=5.5)
        
        cond = "clean"
        if count_en % 6 == 0: 
            arr = apply_phone_bandpass(arr, 16000)
            cond = "phone"
        elif count_en % 6 == 1:
            arr = apply_room_reverb(arr, 16000)
            cond = "reverb"
            
        sf.write(f"eval_2k_dataset/real/en_real_{cond}_{count_en:04d}.wav", arr, 16000)
        count_en += 1
        if count_en % 100 == 0: print(f"   [English Real] Ingested {count_en}/500 clips...")
print(f"✅ 500 Real English clips ready.")

print("\n📥 2. Streaming 500 Real Hindi clips from Google FLEURS test split...")
count_hi = len(glob.glob("eval_2k_dataset/real/hi_real_*.wav"))
if count_hi < 500:
    ds_hi = load_dataset("google/fleurs", "hi_in", split="test", streaming=True)
    skipped = 0
    for item in ds_hi:
        if skipped < 100:
            skipped += 1
            continue
        if count_hi >= 500: break
        arr = item["audio"]["array"]
        sr = item["audio"]["sampling_rate"]
        if sr != 16000: arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
        arr = bound_duration(arr, sr=16000, min_sec=3.0, max_sec=5.5)
        
        cond = "clean"
        if count_hi % 6 == 0:
            arr = apply_phone_bandpass(arr, 16000)
            cond = "phone"
        elif count_hi % 6 == 1:
            arr = apply_room_reverb(arr, 16000)
            cond = "reverb"
            
        sf.write(f"eval_2k_dataset/real/hi_real_{cond}_{count_hi:04d}.wav", arr, 16000)
        count_hi += 1
        if count_hi % 100 == 0: print(f"   [Hindi Real] Ingested {count_hi}/500 clips...")
print(f"✅ 500 Real Hindi clips ready (Total Real: 1,000 clips).")


In [ ]:
# =====================================================
# Step 4: Parallel Synthesis of 1,000 AI Synthetic Audio Clips
# (500 English AI + 500 Hindi AI with multi-speaker vocoders)
# =====================================================
import asyncio
import edge_tts

EN_PROMPTS = [
    "This is an urgent security notification from fraud detection regarding a wire transfer request. Press one to authorize or zero to report.",
    "Your credit card statement indicates suspicious foreign transactions originating from an unregistered terminal. Please authenticate your identity.",
    "Please do not share your confidential six digit one time password or banking PIN with anyone claiming to represent technical support.",
    "Your scheduled express parcel delivery has been held at the regional fulfillment hub due to an incomplete delivery address verification.",
    "We have updated our terms of service and identity management protocol. Please review your email for the detailed documentation.",
    "An unfamiliar device attempted to log into your primary email account from an unrecognized IP address earlier today.",
    "Our synthetic speech intelligence platform models acoustic vocal tract geometry and prosodic intonation with high fidelity.",
    "Thank you for calling customer assistance. All available customer service representatives are currently engaged with other callers.",
    "Your medical consultation has been rescheduled for Thursday morning at nine thirty. Reply directly to this automated prompt to confirm.",
    "Attention: your cloud storage subscription plan will expire within forty-eight hours unless an active renewal is scheduled."
]

HI_PROMPTS = [
    "नमस्ते, यह आपके बैंक के सुरक्षा विभाग से एक स्वचालित कॉल है। कृपया हाल के लेनदेन की पुष्टि करें।",
    "आपके खाते में एक नए अज्ञात उपकरण से लॉग इन करने का प्रयास किया गया था। यदि यह आप नहीं हैं तो तुरंत शून्य दबाएं।",
    "कृपया ध्यान दें कि बैंक का कोई भी प्रतिनिधि आपसे कभी भी आपका गोपनीय एटीएम पिन या ओटीपी पासवर्ड नहीं मांगता है।",
    "ग्राहक सहायता केंद्र में आपका स्वागत है। हमारे सभी सेवा अधिकारी वर्तमान में अन्य कॉलों में व्यस्त हैं, कृपया प्रतीक्षा करें।",
    "आपके पंजीकृत मोबाइल नंबर पर छह अंकों का सुरक्षा सत्यापन कोड भेजा गया है। इसे किसी के साथ साझा न करें।",
    "साइबर सुरक्षा नियमों का पालन करना आपके वित्तीय और व्यक्तिगत डेटा को इंटरनेट पर सुरक्षित रखने के लिए आवश्यक है।",
    "आपकी नई चेकबुक और डेबिट कार्ड आपके पते पर भेज दिया गया है, जो अगले दो कार्य दिवसों में प्राप्त हो जाएगा।",
    "अज्ञात कॉलर द्वारा आपातकालीन धन हस्तांतरण का अनुरोध मिलने पर हमेशा अपने परिजनों से प्रत्यक्ष संपर्क करें।"
]

EN_VOICES = ["en-US-GuyNeural", "en-US-JennyNeural", "en-US-AriaNeural", "en-GB-RyanNeural", "en-IN-NeerjaNeural", "en-US-ChristopherNeural"]
HI_VOICES = ["hi-IN-MadhurNeural", "hi-IN-SwaraNeural"]

sem = asyncio.Semaphore(15)

async def generate_ai_clip(text, voice, out_base, rate="+0%", pitch="+0Hz", condition="clean"):
    wav_file = f"{out_base}.wav"
    if os.path.exists(wav_file): return
    mp3_file = f"{out_base}.mp3"
    async with sem:
        for _ in range(3):
            try:
                comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
                await comm.save(mp3_file)
                arr, _ = librosa.load(mp3_file, sr=16000)
                arr = bound_duration(arr, sr=16000, min_sec=3.0, max_sec=5.5)
                
                if condition == "phone":
                    arr = apply_phone_bandpass(arr, 16000)
                elif condition == "reverb":
                    arr = apply_room_reverb(arr, 16000)
                elif condition == "ambient":
                    arr = apply_ambient_noise(arr, snr_db=22.0)
                    
                sf.write(wav_file, arr, 16000)
                if os.path.exists(mp3_file): os.remove(mp3_file)
                break
            except Exception:
                await asyncio.sleep(1.0)

async def run_ai_synthesis():
    tasks = []
    # 500 English AI
    for i in range(500):
        v = EN_VOICES[i % len(EN_VOICES)]
        p = EN_PROMPTS[i % len(EN_PROMPTS)]
        r = f"{((i % 7) - 3) * 4:+d}%"
        pi = f"{((i % 5) - 2) * 6:+d}Hz"
        cond = "clean"
        if i % 6 == 0: cond = "phone"
        elif i % 6 == 1: cond = "reverb"
        elif i % 6 == 2: cond = "ambient"
        out_base = f"eval_2k_dataset/fake/en_ai_{cond}_{i:04d}"
        tasks.append(generate_ai_clip(p, v, out_base, rate=r, pitch=pi, condition=cond))
        
    # 500 Hindi AI
    for i in range(500):
        v = HI_VOICES[i % len(HI_VOICES)]
        p = HI_PROMPTS[i % len(HI_PROMPTS)]
        r = f"{((i % 5) - 2) * 4:+d}%"
        pi = f"{((i % 5) - 2) * 8:+d}Hz"
        cond = "clean"
        if i % 6 == 0: cond = "phone"
        elif i % 6 == 1: cond = "reverb"
        elif i % 6 == 2: cond = "ambient"
        out_base = f"eval_2k_dataset/fake/hi_ai_{cond}_{i:04d}"
        tasks.append(generate_ai_clip(p, v, out_base, rate=r, pitch=pi, condition=cond))
        
    print(f"🚀 Generating {len(tasks)} AI Synthetic Audio Clips (500 English + 500 Hindi)...")
    await asyncio.gather(*tasks)

await run_ai_synthesis()
print("✅ 1,000 AI Synthetic Audio Clips generated successfully!")


In [ ]:
# =====================================================
# Step 5: High-Speed Batch Inference on NVIDIA T4 GPU
# =====================================================
import time
import torch
import pandas as pd
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

model_id = "ArmanSandhu1729/voiceshield-hindi-english-deepfake-v3-hardened"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🧠 Loading {model_id} onto {device}...")

try:
    extractor = AutoFeatureExtractor.from_pretrained(model_id)
    model = AutoModelForAudioClassification.from_pretrained(model_id).to(device)
except Exception as e:
    print(f"⚠️ v3 model loading exception ({e}), falling back to v2 checkpoint...")
    model_id = "ArmanSandhu1729/voiceshield-hindi-english-deepfake-v2"
    extractor = AutoFeatureExtractor.from_pretrained(model_id)
    model = AutoModelForAudioClassification.from_pretrained(model_id).to(device)

model.eval()

# Resolve fake index
id2label = getattr(model.config, "id2label", {0: "fake", 1: "real"})
fake_idx = 0
for idx, lbl in id2label.items():
    if "fake" in str(lbl).lower() or "ai" in str(lbl).lower() or str(idx) == "1":
        fake_idx = int(idx)
        break
print(f"Model Labels: {id2label} | Using fake_idx = {fake_idx}")

real_files = sorted(glob.glob("eval_2k_dataset/real/*.wav"))
fake_files = sorted(glob.glob("eval_2k_dataset/fake/*.wav"))
eval_items = [(f, "real") for f in real_files] + [(f, "fake") for f in fake_files]

import random
random.seed(42)
random.shuffle(eval_items)

print(f"⚡ Starting Batch GPU Inference across {len(eval_items)} clips...")

results = []
tp, tn, fp, fn = 0, 0, 0, 0
en_total, en_correct = 0, 0
hi_total, hi_correct = 0, 0
cond_stats = {}
latencies = []

t_eval_start = time.time()

with torch.no_grad():
    for idx, (fpath, ground_truth) in enumerate(eval_items, 1):
        fname = os.path.basename(fpath)
        lang = "hindi" if "hi_" in fname else "english"
        cond = "clean"
        if "phone" in fname: cond = "phone_bandpass"
        elif "reverb" in fname: cond = "reverb_replay"
        elif "ambient" in fname: cond = "ambient_noise"
        
        speech, sr = sf.read(fpath)
        dur = len(speech) / sr
        
        t0 = time.time()
        inputs = extractor(speech, sampling_rate=16000, return_tensors="pt").to(device)
        logits = model(**inputs).logits.squeeze()
        probs = torch.softmax(logits / 2.0, dim=-1).cpu().tolist()
        lat = (time.time() - t0) * 1000.0
        latencies.append(lat)
        
        fake_prob = probs[fake_idx]
        pred = "AI_GENERATED" if fake_prob >= 0.50 else "HUMAN_SPEECH"
        
        is_correct = False
        if ground_truth == "fake":
            if pred == "AI_GENERATED":
                tp += 1; is_correct = True
            else: fn += 1
        else:
            if pred == "HUMAN_SPEECH":
                tn += 1; is_correct = True
            else: fp += 1
            
        if lang == "hindi":
            hi_total += 1
            if is_correct: hi_correct += 1
        else:
            en_total += 1
            if is_correct: en_correct += 1
            
        if cond not in cond_stats: cond_stats[cond] = {"total": 0, "correct": 0}
        cond_stats[cond]["total"] += 1
        if is_correct: cond_stats[cond]["correct"] += 1
        
        results.append({
            "filename": fname,
            "language": lang,
            "condition": cond,
            "ground_truth": ground_truth,
            "prediction": pred,
            "is_correct": is_correct,
            "fake_probability": round(fake_prob, 4),
            "duration_sec": round(dur, 2),
            "latency_ms": round(lat, 1)
        })
        
        if idx % 200 == 0 or idx == len(eval_items):
            curr_acc = ((tp + tn) / len(results)) * 100.0
            print(f"[{idx:4d}/{len(eval_items)}] Processed {fname[:32]:32s} | Live Acc: {curr_acc:5.2f}% | Latency: {lat:.1f}ms")

total_elapsed = time.time() - t_eval_start
print(f"\n✅ Finished full inference run in {total_elapsed:.1f} seconds!")


In [ ]:
# =====================================================
# Step 6: Empirical Statistical Scorecard & Metrics
# =====================================================
total = len(results)
accuracy = ((tp + tn) / total) * 100.0
precision = (tp / (tp + fp) * 100.0) if (tp + fp) > 0 else 0.0
recall = (tp / (tp + fn) * 100.0) if (tp + fn) > 0 else 0.0
specificity = (tn / (tn + fp) * 100.0) if (tn + fp) > 0 else 0.0
f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
fpr = (fp / (fp + tn) * 100.0) if (fp + tn) > 0 else 0.0
fnr = (fn / (fn + tp) * 100.0) if (fn + tp) > 0 else 0.0

en_acc = (en_correct / en_total * 100.0) if en_total > 0 else 0.0
hi_acc = (hi_correct / hi_total * 100.0) if hi_total > 0 else 0.0
avg_lat = np.mean(latencies)

print("=" * 80)
print("📊 VOICESHIELD 2,000-SAMPLE RIGOROUS BENCHMARK SCORECARD")
print("=" * 80)
print(f"Model Evaluated:             {model_id}")
print(f"Total Test Audio Clips:      {total} (1,000 Real Human + 1,000 AI Synthetic)")
print(f"Average Duration:            {np.mean([r['duration_sec'] for r in results]):.2f} seconds")
print(f"Average Latency on T4 GPU:   {avg_lat:.1f} ms / sample")
print("-" * 80)
print(f"🎯 Overall Accuracy:         {accuracy:.2f}%  ({tp+tn}/{total})")
print(f"🔍 Precision (AI):           {precision:.2f}%")
print(f"📈 Recall / Sensitivity:     {recall:.2f}%")
print(f"🛡️ Specificity (Human):      {specificity:.2f}%")
print(f"⚡ F1-Score:                 {f1:.2f}%")
print(f"⚠️ False Positive Rate (FPR): {fpr:.2f}%  (Human caller falsely flagged as AI)")
print(f"⚠️ False Negative Rate (FNR): {fnr:.2f}%  (AI cloned voice missed as Human)")
print("-" * 80)
print(f"🌐 English Accuracy:         {en_acc:.2f}%  ({en_correct}/{en_total})")
print(f"🌐 Hindi Accuracy:           {hi_acc:.2f}%  ({hi_correct}/{hi_total})")
print("-" * 80)
print("Acoustic Channel Stress Breakdown:")
for c_name, st in cond_stats.items():
    c_rate = (st['correct'] / st['total']) * 100.0 if st['total'] > 0 else 0.0
    print(f"  • {c_name:18s}: {c_rate:5.2f}% ({st['correct']}/{st['total']})")
print(f"Confusion Matrix:            [True Positive: {tp}, False Positive: {fp}, False Negative: {fn}, True Negative: {tn}]")
print("=" * 80)

# Export Row-level telemetry CSV
df_out = pd.DataFrame(results)
df_out.to_csv("voiceshield_2k_benchmark_predictions.csv", index=False)
print("\n📁 Exported complete predictions log to: voiceshield_2k_benchmark_predictions.csv")


In [ ]:
# =====================================================
# Step 7: Automated Quality Gate & Golden Vault Ingestion
# Folder Name: test-Bi-new-2k-01
# =====================================================
import shutil
import json
from google.colab import files

TARGET_FOLDER = "test-Bi-new-2k-01"

# Realistic certification criteria (Robust, not artificial 100%)
PASS_THRESHOLD_ACC = 88.0
PASS_THRESHOLD_PREC = 85.0

if accuracy >= PASS_THRESHOLD_ACC and precision >= PASS_THRESHOLD_PREC:
    print(f"🎉 MODEL CERTIFICATION PASSED! (Accuracy: {accuracy:.2f}% >= {PASS_THRESHOLD_ACC}%, Precision: {precision:.2f}% >= {PASS_THRESHOLD_PREC}%)")
    print(f"📦 Creating Golden Vault archive: '{TARGET_FOLDER}'...")
    
    os.makedirs(f"{TARGET_FOLDER}/real", exist_ok=True)
    os.makedirs(f"{TARGET_FOLDER}/fake", exist_ok=True)
    
    # Copy files into structured vault
    for f in glob.glob("eval_2k_dataset/real/*.wav"):
        shutil.copy2(f, f"{TARGET_FOLDER}/real/")
    for f in glob.glob("eval_2k_dataset/fake/*.wav"):
        shutil.copy2(f, f"{TARGET_FOLDER}/fake/")
        
    # Write metadata manifest
    manifest = {
        "vault_name": TARGET_FOLDER,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "total_audio_samples": total,
        "real_samples": len(real_files),
        "fake_samples": len(fake_files),
        "tested_model": model_id,
        "empirical_metrics": {
            "accuracy_percent": round(accuracy, 2),
            "precision_percent": round(precision, 2),
            "recall_percent": round(recall, 2),
            "f1_percent": round(f1, 2),
            "specificity_percent": round(specificity, 2),
            "false_positive_rate_percent": round(fpr, 2),
            "false_negative_rate_percent": round(fnr, 2),
            "english_accuracy_percent": round(en_acc, 2),
            "hindi_accuracy_percent": round(hi_acc, 2)
        }
    }
    with open(f"{TARGET_FOLDER}/metadata.json", "w") as f:
        json.dump(manifest, f, indent=2)
    
    # Copy prediction csv inside folder
    shutil.copy2("voiceshield_2k_benchmark_predictions.csv", f"{TARGET_FOLDER}/")
    
    # Create zip archive
    print(f"Compressing '{TARGET_FOLDER}' into {TARGET_FOLDER}.zip...")
    shutil.make_archive(TARGET_FOLDER, 'zip', TARGET_FOLDER)
    print(f"✅ Successfully packaged {TARGET_FOLDER}.zip!")
    
    print("\n⬇️ Triggering automated download of benchmark dataset zip...")
    files.download(f"{TARGET_FOLDER}.zip")
    print(f"💡 After downloading, extract the zip into: VoiceCloning26104/samples_golden_vault/{TARGET_FOLDER}")
else:
    print(f"⚠️ Benchmark did not reach the threshold (Acc: {accuracy:.2f}%, Prec: {precision:.2f}%). Review CSV predictions before archiving.")
